In [ ]:
# Data: ../csv/Fig2_E_replay_non_replay/  (the original loading code is kept below each data line, as comments)
import os, glob, pickle, re, sys   # standard imports
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'analysis_scripts'))   # the criticality library
import pandas as pd
import numpy as np
CSV = Path('..') / 'csv' / 'Fig2_D_E_replayprobality_SWR_replaynon_replay'
OUT = Path('..') / 'figure_output'
OUT.mkdir(exist_ok=True)


def read_csv(name, **kw):
    """A table of this notebook (round_trip: the floats exactly as written)."""
    return pd.read_csv(CSV / name, float_precision='round_trip', **kw)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import sem
import os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress, spearmanr
import statsmodels.formula.api as smf
from sklearn.metrics import r2_score
import matplotlib as mpl
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans', 'sans-serif']

# Make mathtext look like Arial
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Arial'
mpl.rcParams['mathtext.it'] = 'Arial:italic'
mpl.rcParams['mathtext.bf'] = 'Arial:bold' 

# (optional) sizes + minus sign
mpl.rcParams['font.size'] = 16
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams['axes.unicode_minus'] = False

# Global sizes (tweak to taste)
mpl.rcParams['font.size'] = 14
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5
# --------- Config ---------
in_dir = Path(r"D:\Documents\GitHub\Sleep_criticaltiy\data\ripple_barr\replay_non_replay")
n_bins = 6
labels6 = [f"Q{i}" for i in range(1, n_bins + 1)]

# === equal-width tiling helper (per-file) ===
import numpy as np
import pandas as pd

def make_tiles(
    x: pd.Series,
    q: int = 6,
    labels: list[str] | None = None,
    mode: str = "equal",                # "equal" or "quantile"
    edges: np.ndarray | None = None,    # optional precomputed edges (for "equal")
    include_lowest: bool = True,
    return_edges: bool = False
):
    """
    Robust tiler with two modes:
      - mode="equal": equal-width bins via pd.cut (optionally with given edges)
      - mode="quantile": equal-count bins via pd.qcut (with duplicates=drop fallback)
    
    Handles constant vectors by assigning all to Q1.
    Returns (tiles, tile_num, edges_or_None) if return_edges=True, else (tiles, tile_num).
    """
    x = pd.Series(x)
    if labels is None:
        labels = [f"Q{i}" for i in range(1, q + 1)]

    # constant or all-NaN -> put everything in Q1
    if x.nunique(dropna=True) <= 1:
        tiles = pd.Series(["Q1"] * len(x), index=x.index, dtype="category")
        tile_num = pd.Series(np.ones(len(x), int), index=x.index)
        return (tiles, tile_num, None) if return_edges else (tiles, tile_num)

    if mode.lower() == "equal":
        # Use provided edges or compute from data
        if edges is None:
            v = x.to_numpy(dtype=float)
            v = v[np.isfinite(v)]
            vmin, vmax = np.nanmin(v), np.nanmax(v)
            if vmin == vmax:
                eps = 1e-9 if vmin == 0 else 1e-6 * abs(vmin)
                vmin -= eps; vmax += eps
            edges = np.linspace(vmin, vmax, q + 1)
        tiles = pd.cut(x, bins=edges, labels=labels, include_lowest=include_lowest)
        tiles = tiles.astype("category")
        tile_num = tiles.cat.codes + 1

        return (tiles, tile_num, edges) if return_edges else (tiles, tile_num)

    elif mode.lower() == "quantile":
        # Try qcut with duplicates=drop; if it collapses, fall back to rank-based qcut
        try:
            tiles = pd.qcut(x, q=q, labels=labels, duplicates="drop")
            # if duplicates dropped, we may have < q bins → relabel to Q1..Qk
            if tiles.cat.categories.size != q:
                # remap to Q1..Qk in order
                k = tiles.cat.categories.size
                new_labels = [f"Q{i}" for i in range(1, k + 1)]
                tiles = tiles.cat.rename_categories(new_labels)
        except Exception:
            # rank-based fallback (stable ordering)
            rk = x.rank(method="first")
            uniq = int(min(q, rk.nunique()))
            if uniq < 1:
                tiles = pd.Series(["Q1"] * len(x), index=x.index, dtype="category")
            else:
                tiles = pd.qcut(rk, q=uniq,
                                labels=[f"Q{i}" for i in range(1, uniq + 1)])
        tiles = tiles.astype("category")
        tile_num = tiles.cat.codes + 1
        # no edges concept for qcut; return None
        return (tiles, tile_num, None) if return_edges else (tiles, tile_num)

    else:
        raise ValueError("mode must be 'equal' or 'quantile'")


# === collect per-row tiles & pool ===
# ---- data (../csv) ----
rows = []
pooled = read_csv('d2_by_replay_nonreplay_swr_rate_tiles.csv')
# ---- the same from the recordings, with analysis_scripts/: ----
# rows = []
# files = sorted(in_dir.glob("*.pkl"))
# if not files:
#     raise FileNotFoundError(f"No .pkl files in {in_dir}")
# 
# for f in files:
#     try:
#         df = pd.read_pickle(f)
#     except Exception as e:
#         print(f"[skip] {f.name}: {e}"); 
#         continue
# 
#     need = {"d2_6tile", "ripple_rate_in", "ripple_rate_out"}
#     if not need.issubset(df.columns):
#         print(f"[skip] {f.name}: missing {need - set(df.columns)}"); 
#         continue
# 
#     d = df[["d2_6tile", "d2_err", "ripple_rate_in", "ripple_rate_out"]].copy()
#     d["file_id"] = f.name
# 
#     # z-score D² within file
#     mu = d["d2_6tile"].mean(skipna=True)
#     sd = d["d2_6tile"].std(skipna=True, ddof=0)
#     sd = sd if np.isfinite(sd) and sd > 0 else 1.0
#     d["d2_z"] = (d["d2_6tile"] - mu) / sd
# 
#     # ---- config for tiling ----
#     TILE_MODE    = "equal"      # "equal" or "quantile"
#     SHARE_EDGES  = True         # only applies when TILE_MODE == "equal"
# 
#     # IN rows
#     sub_in = d[["file_id", "d2_z", "d2_err", "ripple_rate_in"]].dropna(subset=["ripple_rate_in"]).copy()
#     edges_this_file = None
#     if not sub_in.empty:
#         tiles_in, tile_num_in, edges_in = make_tiles(
#             sub_in["ripple_rate_in"], q=n_bins, labels=labels6,
#             mode=TILE_MODE, return_edges=True
#         )
#         sub_in["tile"] = tiles_in
#         sub_in["tile_num"] = tile_num_in
#         sub_in["group"] = "IN"
#         sub_in = sub_in.rename(columns={"ripple_rate_in": "rate"})
#         rows.append(sub_in[["file_id","group","tile","tile_num","d2_z","d2_err","rate"]])
#         # keep edges for OUT if sharing (equal-width only)
#         if TILE_MODE == "equal" and SHARE_EDGES:
#             edges_this_file = edges_in
# 
#     # OUT rows
#     sub_out = d[["file_id", "d2_z", "d2_err", "ripple_rate_out"]].dropna(subset=["ripple_rate_out"]).copy()
#     if not sub_out.empty:
#         tiles_out, tile_num_out, _ = make_tiles(
#             sub_out["ripple_rate_out"], q=n_bins, labels=labels6,
#             mode=TILE_MODE,
#             edges=edges_this_file if (TILE_MODE == "equal" and SHARE_EDGES and edges_this_file is not None) else None,
#             return_edges=True
#         )
#         sub_out["tile"] = tiles_out
#         sub_out["tile_num"] = tile_num_out
#         sub_out["group"] = "OUT"
#         sub_out = sub_out.rename(columns={"ripple_rate_out": "rate"})
#         rows.append(sub_out[["file_id","group","tile","tile_num","d2_z","d2_err","rate"]])
# 
# # pooled, all rows
# pooled = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
if pooled.empty:
    raise ValueError("No usable rows after tiling.")

# keep 1..6 just in case
pooled = pooled[pooled["tile_num"].between(1, n_bins)]

# === ALL-ROWS summary: mean ± SEM across rows (not cross-file) ===
g = pooled.groupby(["group","tile_num"], observed=True)["d2_z"]
means = g.mean().unstack("group").reindex(range(1, n_bins+1))
errs  = g.apply(lambda x: sem(x, nan_policy="omit")).unstack("group").reindex(range(1, n_bins+1))
from scipy.stats import sem

# === 1) Mean across all rows (same as before) ===
g_rows = pooled.groupby(["group","tile_num"], observed=True)["d2_z"]
means  = g_rows.mean().unstack("group").reindex(range(1, n_bins+1))

# === 2) SEM across files (file-level means) ===
# First compute mean d2_z per (file, group, tile_num)
file_means = (
    pooled
    .groupby(["file_id", "group", "tile_num"], observed=True)["d2_z"]
    .mean()
    .reset_index()
)

# Now take SEM across files for each (group, tile_num)
g_files = file_means.groupby(["group", "tile_num"], observed=True)["d2_z"]
errs = g_files.apply(lambda x: sem(x, nan_policy="omit")).unstack("group").reindex(range(1, n_bins+1))

# === plot mean ± SEM across rows ===
fig, ax = plt.subplots(figsize=(4,4))
xs = np.arange(1, n_bins+1)

y_in  = means.get("IN")
e_in  = errs.get("IN")
y_out = means.get("OUT")
e_out = errs.get("OUT")

ax.errorbar(xs, y_in.values,  yerr=e_in.values,  fmt='o-', capsize=4, lw=2, label=None,color="#691c1bc0")
ax.errorbar(xs, y_out.values, yerr=e_out.values, fmt='s--', capsize=4, lw=2, label=None,color="#d62728ff")

xs = np.arange(1, n_bins+1)

ax.set_xticks(xs)                                  # positions: 1..6
ax.set_xticklabels([str(i) for i in xs])           # labels: "1".."6"
# or, with newer Matplotlib:
# ax.set_xticks(xs, labels=[str(i) for i in xs])

ax.set_xlabel("Ripple rate equal-width tiles (low → high)")
ax.set_ylabel("d2 distance (z)")

for s in ("top","right"): ax.spines[s].set_visible(False)
ax.legend(frameon=False)

plt.tight_layout()
save_path = r"../figure_output\fig2_E_d2_vs_ripple_with replay_ripple_not_replay.svg"
plt.savefig(save_path, dpi=300)
plt.show()

In [ ]:
# ============================
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import r2_score
import statsmodels.api as sm
# If file_id looks like "PPP8_day8_CA1.pkl"
pooled = pooled.copy()
pooled["animal_id"] = pooled["file_id"].astype(str).str.split("_").str[0]
# Nesting key (safe even if file_id repeats across animals)
pooled["file_in_animal"] = pooled["animal_id"].astype(str) + "|" + pooled["file_id"].astype(str)

import numpy as np

def pfmt(p):
    if p is None or (isinstance(p, float) and not np.isfinite(p)):
        return "NA"
    if p == 0.0:
        return "<1e-15"   # reporting floor; change if you want
    if p < 1e-3:
        return f"{p:.2e}"
    return f"{p:.3f}"


import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import norm
from sklearn.metrics import r2_score

from statsmodels.stats.sandwich_covariance import cov_cluster

def fit_one_group_LMM(df_long, group_name, xname="tile_num", yname="d2_z", n_bins=None):
    """
    Fit: y ~ x
    Random intercepts:
      - animal_id (groups)
      - file_in_animal (variance component; nested session/file)

    Also reports a cluster-robust p-value for the slope clustered by animal_id.
    """

    d = df_long.copy()
    d = d[d["group"] == group_name]

    if n_bins is not None:
        d = d[d[xname].between(1, n_bins)]

    # Require IDs
    need = [yname, xname, "file_id", "animal_id", "file_in_animal"]
    d = d.dropna(subset=need)

    n_animals = d["animal_id"].nunique()
    n_files   = d["file_in_animal"].nunique()
    if n_animals < 2 or n_files < 2 or len(d) < 6:
        return {"group": group_name, "ok": False, "msg": "insufficient data"}

    try:
        # Random intercept per animal via groups=animal_id
        # Random intercept per file (nested) via variance components
        vc = {"file_in_animal": "0 + C(file_in_animal)"}

        m = smf.mixedlm(
            f"{yname} ~ {xname}",
            data=d,
            groups=d["animal_id"],
            vc_formula=vc,
            re_formula="1",
        )

        r = m.fit(method="lbfgs", reml=False, disp=False)

        # Fixed effects
        fe = r.fe_params
        slope = float(fe.get(xname, np.nan))

        # Default (model-based) SE / p (not robust)
        se_default = float(r.bse_fe.get(xname, np.nan)) if hasattr(r, "bse_fe") else float(r.bse.get(xname, np.nan))
        p_default  = float(r.pvalues.get(xname, np.nan))

        # Cluster-robust covariance of fixed effects clustered by animal_id
        # This gives robust SE/p for Intercept and slope (fixed effects only)
        p_robust = np.nan
        se_robust = np.nan
        try:
            cov_fe_rob = cov_cluster(r, d["animal_id"])  # robust cov for parameters
            # cov_cluster returns cov for ALL params in r.params; we only want fixed effects
            # Fixed effects are first k_fe entries
            k_fe = len(r.fe_params)
            cov_fe_rob = np.asarray(cov_fe_rob)[:k_fe, :k_fe]

            se_vec = np.sqrt(np.diag(cov_fe_rob))
            fe_names = list(r.fe_params.index)

            if xname in fe_names:
                j = fe_names.index(xname)
                se_robust = float(se_vec[j])
                z = slope / se_robust if se_robust > 0 else np.nan
                p_robust = float(2 * norm.sf(abs(z)))  # two-sided robust p
        except Exception:
            pass

        # Fixed-effects-only R2 (same as your approach)
        yhat_fe = fe["Intercept"] + fe[xname] * d[xname].values
        r2_fe = r2_score(d[yname], yhat_fe)

        return {
            "group": group_name, "ok": True,
            "slope": slope,
            "se_default": se_default, "p_default": p_default,
            "se_robust": se_robust, "p_robust": p_robust,
            "r2_fe": float(r2_fe),
            "n_rows": int(len(d)),
            "n_animals": int(n_animals),
            "n_files": int(n_files),
            "model": r,
        }

    except Exception as e:
        return {"group": group_name, "ok": False, "msg": f"MixedLM failed: {e}"}
res_in  = fit_one_group_LMM(pooled, "IN",  xname="tile_num", yname="d2_z", n_bins=n_bins)
res_out = fit_one_group_LMM(pooled, "OUT", xname="tile_num", yname="d2_z", n_bins=n_bins)

print("\n===== Linear fit per condition (animal + file nested) =====")
for res in (res_in, res_out):
    if not res["ok"]:
        print(f"{res['group']}: {res['msg']}")
        continue

    # prefer robust if available
    p_use  = res["p_robust"] if np.isfinite(res["p_robust"]) else res["p_default"]
    se_use = res["se_robust"] if np.isfinite(res["se_robust"]) else res["se_default"]
    tag = res["group"]
    p_tag = "p_rob" if np.isfinite(res["p_robust"]) else "p"

    print(
        f"{tag}: {p_tag}={pfmt(p_use)}, "
        f"animals={res['n_animals']}, files={res['n_files']}"
    )